In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
df['TotalCharges'] = df['TotalCharges'].str.strip().replace('', '0')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df.shape

(7043, 21)

In [2]:
df = df.drop(columns=['customerID'])

In [3]:
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [4]:
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 60, np.inf],
    labels=['0-1yr', '1-2yr', '2-4yr', '4-5yr', '5yr+']
)

addon_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

def count_addons(row):
    return sum(1 for col in addon_services if row[col] == 'Yes')

df['num_addon_services'] = df.apply(count_addons, axis=1)

df['avg_monthly_spend'] = df['TotalCharges'] / df['tenure'].replace(0, 1)

In [5]:
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                   'Contract', 'PaymentMethod', 'tenure_group']
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_addon_services', 'avg_monthly_spend']

In [6]:
for col in binary_cols:
    df[col] = (df[col] == df[col].unique()[0]).astype(int) if df[col].nunique() == 2 else df[col]

binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
for col in binary_cols:
    df[col] = df[col].map(binary_map).fillna(df[col]).astype(int)

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

df.shape

(7043, 37)

In [7]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())

Train shape: (5634, 36)
Test shape: (1409, 36)
Train churn rate: 0.2653532126375577
Test churn rate: 0.2654364797728886


In [8]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Processed data saved.")

Processed data saved.
